In [14]:
import torch
from pathlib import Path

@torch.no_grad()
def compute_fid_stats_from_pt(shards_dir, pattern="feats_*.pt", save_pt=None):
    paths = sorted(Path(shards_dir).glob(pattern))
    if not paths: raise FileNotFoundError(f"No shards in {shards_dir} matching {pattern}")
    n, mu, S = 0, None, None
    for p in paths:
        X = torch.load(p, map_location="cpu")["feats"].to(torch.float64)  # (m,d)
        m = X.size(0)
        if m == 0: continue
        mu_b = X.mean(0)
        Xc   = X - mu_b
        Cb   = Xc.t() @ Xc
        if mu is None:
            d  = X.size(1)
            mu = torch.zeros(d, dtype=torch.float64)
            S  = torch.zeros(d, d, dtype=torch.float64)
        Δ   = mu_b - mu
        mu  = mu + Δ * (m / (n + m))
        S   = S + Cb + torch.outer(Δ, Δ) * (n * m / (n + m))
        n  += m
    if n < 2: raise ValueError(f"Too few samples: n={n}")
    sigma = S / (n - 1)
    if save_pt: torch.save({"mu": mu, "sigma": sigma, "n": n}, save_pt)
    return mu, sigma, n


In [15]:
mu, sigma, n = compute_fid_stats_from_pt(
    "/home/scpark/data/imagenet_feats/train_clean",
    pattern="feats_*.pt",
    save_pt="/home/scpark/data/imagenet_feats/train_clean_stats.pt"
)


In [17]:
n

1281167

In [13]:
exts = {".jpg",".jpeg",".png",".bmp",".webp"}
in_root="/home/scpark/data/imagenet/train"
all_paths = sorted(p for p in Path(in_root).rglob("*") if p.is_file() and p.suffix.lower() in exts)
len(all_paths)

1281167